# Logistic Regression on the Full Dataset

This notebook is more for exploratory purposes. We want to run Logistic Regression on the entire dataset to see how well it performs if no feature selection was used. 

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV

from sklearn.metrics import accuracy_score

import matplotlib.pyplot as plt

In [ ]:
path = ['./Data/X_train.csv', './Data/y_train.csv']
X_train, y_train = [pd.read_csv(f, index_col=0) for f in path] 

In [ ]:
display(X_train.head())
display(y_train.head())
y_train= np.array(y_train['Cluster'])

In [ ]:
y_train

## Logistic Regression on full training dataset


In [ ]:
log_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=10000))
])

scores = cross_val_score(log_pipe, X_train, y_train, cv=5, scoring='accuracy')
print(scores)
print(f"CV Accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

# [0.87704918 0.85245902 0.91803279 0.92622951 0.86885246]
# CV Accuracy: 0.889 ± 0.029

In [ ]:
# Define pipeline
pipe = Pipeline([
    ('scale', StandardScaler()),
    ('pca', PCA(n_components=2)),
    ('model', LogisticRegression(max_iter=50000))
])


# Define parameter grid for tuning
param_grid = {
    'pca__n_components': [2, 10, 20, 60, 100],
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100],  # inverse regularization strength
    'model__penalty': ['l1', 'l2'],              # can test 'l1' if using solver='liblinear'
    'model__solver': ['liblinear', 'saga'],
    'model__max_iter': [200000]
}

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV will find the best combo of params
grid_search = GridSearchCV(pipe, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best parameters found:")
print(grid_search.best_params_)

print("\nBest cross-validation accuracy:")
print(grid_search.best_score_)

final_model = grid_search.best_estimator_

# Best parameters found:
# {'model__C': 0.1, 'model__max_iter': 200000, 'model__penalty': 'l1', 'model__solver': 'saga'}

# Best cross-validation accuracy:
# 0.9229508196721312
